# Zenith on Colab

Pulls the latest `rbf22/zenith` from GitHub and runs the experiment set in the settings cell.

**One-time setup**
1. Open this notebook from GitHub (`File > Open notebook > GitHub`, `rbf22/zenith`).
2. Runtime > Change runtime type > GPU.
3. For the resume experiment only: upload `ckpt_step9000/` to `MyDrive/zenith/`. Nothing else needs uploading: the tokenizer comes from the repo, and the data is generated from public Hugging Face datasets.

T4 has no bf16 and runs fp32; L4 or A100 run bf16.

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
from google.colab import drive, userdata
import base64, subprocess, os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/zenith'
DATA = f'{DRIVE}/zenith_data'
REPO = '/content/zenith'
REMOTE = 'https://github.com/rbf22/zenith.git'
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
header = ['-c', f'http.extraheader=Authorization: basic {base64.b64encode(f"x-access-token:{token}".encode()).decode()}'] if token else []
def git(*args, cwd=None):
    subprocess.run(['git', *header, *args], cwd=cwd, check=True)
if os.path.exists(f'{REPO}/.git'):
    git('fetch', '-q', 'origin', cwd=REPO)
    git('reset', '-q', '--hard', 'origin/main', cwd=REPO)
else:
    try:
        git('clone', '-q', REMOTE, REPO)
    except subprocess.CalledProcessError:
        raise SystemExit('Clone failed. The repo is private: add a GITHUB_TOKEN secret (read access to rbf22/zenith), or make the repo public.')
print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO, capture_output=True, text=True).stdout)

In [ ]:
!pip install -q transformers datasets tokenizers pytest pytest-cov

In [ ]:
%cd /content/zenith
import os, subprocess
TOK = '/content/zenith/tokenizer32k'
os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f'{DATA}/train.jsonl'):
    subprocess.run(['python', 'scripts/prepare_data.py', '--out_dir', DATA, '--train_chars', '150001554', '--tokenizer_chars', '100000000'], check=True)
if not os.path.exists(f'{DATA}/heldout.jsonl'):
    subprocess.run(['python', 'scripts/make_heldout.py', '--out', f'{DATA}/heldout.jsonl', '--docs', '400'], check=True)
print({p: os.path.exists(p) for p in [TOK, f'{DATA}/train.jsonl', f'{DATA}/heldout.jsonl']})
assert os.path.isdir(TOK) and os.path.exists(f'{DATA}/train.jsonl') and os.path.exists(f'{DATA}/heldout.jsonl')

Data is generated once into Drive (about 10 to 20 minutes, streaming from Hugging Face). Later runs reuse it.


In [ ]:
%cd /content/zenith
!python -m pytest -q --cov=zenith --cov-report=term-missing 2>&1 | tail -15

## Settings

Choose the experiment here and re-run the cell below. Results go to Google Drive, so they survive session timeouts.

In [ ]:
EXPERIMENT = 'lr_sweep'          # 'lr_sweep' or 'main_resume'
SWEEP_LRS = ['2e-4', '1e-4']    # 300-step runs; '5e-4' already done (held-out 7.54), '2e-4' done (7.30)
MAIN_LR = '2e-4'                # peak LR for main_resume
MAIN_STEPS = 33000
RESULTS = f'{DRIVE}/results'

In [ ]:
%cd /content/zenith
import shutil, subprocess
os.makedirs(RESULTS, exist_ok=True)
def train_cmd(out, lr, steps, warmup, save, log):
    return ['python', '-m', 'zenith.train', '--config', 'configs/target/config.json', '--tokenizer', TOK,
            '--data', f'{DATA}/train.jsonl', '--output_dir', out, '--seq_len', '512', '--batch_size', '2',
            '--max_steps', str(steps), '--warmup_steps', str(warmup), '--lr', lr, '--save_steps', str(save),
            '--log_steps', str(log), '--compile_scan']
def heldout(ckpt, out):
    subprocess.run(['python', 'scripts/eval_ckpt.py', '--checkpoints', ckpt, '--tokenizer', TOK,
                    '--heldout', f'{DATA}/heldout.jsonl', '--max_windows', '128'], stdout=open(out, 'w'), stderr=subprocess.STDOUT, check=True)
    return open(out).read().strip().splitlines()[-1]
if EXPERIMENT == 'lr_sweep':
    for lr in SWEEP_LRS:
        out = f'{RESULTS}/sweep_lr{lr}'
        if os.path.exists(f'{out}/eval.log'):
            print(lr, 'already done'); continue
        os.makedirs(out, exist_ok=True)
        subprocess.run(train_cmd(f'{out}/ckpt', lr, 300, 30, 100000, 25), stdout=open(f'{out}/train.log', 'w'), stderr=subprocess.STDOUT, check=True)
        print(lr, heldout(f'{out}/ckpt', f'{out}/eval.log'))
elif EXPERIMENT == 'main_resume':
    run = f'{RESULTS}/main_lr{MAIN_LR}/ckpt'
    if not os.path.exists(run):
        shutil.copytree(f'{DRIVE}/ckpt_step9000', run)
    subprocess.run(train_cmd(run, MAIN_LR, MAIN_STEPS, 1000, 1000, 50), check=True)
else:
    raise ValueError(EXPERIMENT)

In [ ]:
%cd /content/zenith
print(heldout(f'{DRIVE}/ckpt_step9000', f'{RESULTS}/eval_step9000.log'))